<a href="https://colab.research.google.com/github/code-sagar-03/csot-ml-astronomy/blob/main/submissions/himanshu_sagar/week2/week2_baselines_and_fcn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# week 2 :- baselines & fully- connected networks

# part 1
   baseline with scikit- learn

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:


from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.path.exists('/content/drive/MyDrive/galaxy_data'))

True


In [ ]:
from torchvision import transforms
from torchvision.datasets import ImageFolder

transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
dataset = ImageFolder(
    '/content/drive/MyDrive/galaxy_data',
    transform=transform
)

print("Classes:", dataset.classes)
print("Total images:", len(dataset))

Classes: ['elliptical', 'spiral', 'spiral_barred']
Total images: 200


# step 1 :- creating a data loader



In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

images, labels = next(iter(loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([32, 3, 64, 64])
Labels shape: torch.Size([32])


# #observation
   the dataloader successfully loads the galaxy images in batches of 32 . each image has 3 channels with a size of 64 x 64 pixels.

# converting tensors to numpy arrays

In [ ]:
images_numpy = images.numpy()
labels_numpy = labels.numpy()

print("Images NumPy shape:", images_numpy.shape)
print("Labels NumPy shape:", labels_numpy.shape)

Images NumPy shape: (32, 3, 64, 64)
Labels NumPy shape: (32,)


# flattering the images

In [ ]:
flattened_images = images_numpy.reshape(images_numpy.shape[0], -1)

print("Original shape:", images_numpy.shape)
print("Flattened shape:", flattened_images.shape)

Original shape: (32, 3, 64, 64)
Flattened shape: (32, 12288)


# #observation
  the images were reshaped into one - dimensional vectors

# splitting the dataset

In [ ]:
from sklearn.model_selection import train_test_split

X = flattened_images
y = labels_numpy

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

Training samples: (25, 12288)
Testing samples: (7, 12288)


# training a baseline classifier

A K- Nearest neighbour(KNN) classifier is trained on the flattened galaxy images to establish a simple baseline performance.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3)

knn.fit(X_train, y_train)

print("Baseline model training completed.")

Baseline model training completed.


 The KNN classifier was successfully trained using the falttened images vector . the model will serve as a baseline for comparision with neutral network approaches

 # evaluating the baseline model


the trained model is evaluated on the test set to measure its classification accuracy

In [ ]:
from sklearn.metrics import accuracy_score

predictions = knn.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Baseline Accuracy:", accuracy)

Baseline Accuracy: 0.7142857142857143


## Why does the baseline struggle?

Flattening converts images into one-dimensional vectors and removes important spatial relationships between pixels. As a result, the classifier cannot effectively capture complex galaxy structures such as spiral arms or central bars.

#                                               part 2
  
   fully connected network (nn.Module)

In [ ]:
import torch
import torch.nn as nn

# building a fully connected network

a simple neutral network is created using pytorch . the model contains two fully connected layers and uses ReLU activation.

In [ ]:
class GalaxyFCN(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(3 * 64 * 64, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        x = x.view(x.size(0), -1)   # flatten images
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x

## Creating the model

The fully connected network is initialized and its architecture is displayed to verify the layer structure.

In [ ]:
model = GalaxyFCN()

print(model)

GalaxyFCN(
  (fc1): Linear(in_features=12288, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)


### Observation

The model architecture contains two fully connected layers with a ReLU activation function in between. The output layer has three neurons corresponding to the three galaxy classes.

In [ ]:
outputs = model(images)

print("Input shape :", images.shape)
print("Output shape:", outputs.shape)

Input shape : torch.Size([32, 3, 64, 64])
Output shape: torch.Size([32, 3])


### Observation

A batch of images was successfully passed through the network. The output tensor contains predictions for the three galaxy classes for each image in the batch.

## Reflection

This week helped me understand the difference between traditional machine learning and neural network approaches. I learned how image data can be flattened for baseline models and how fully connected networks can be built using PyTorch. Passing a batch through the model also helped me understand how data flows through different layers of a neural network.